# IRIS Myopia Classifier — Vinay Run
**Google Colab Experiment Notebook**

Run cells top to bottom. Each section is one step of the experiment plan.

---
## Before you start (VS Code + Colab extension)

1. Open this notebook in VS Code/Cursor
2. **Select Kernel** (top right) → **Colab** → **New Colab Server** → pick **GPU (T4)**
3. Sign in with Google when prompted
4. **Upload scripts to Colab:** right-click the `vinay-run/` folder in Explorer → **Upload to Colab**
   - This uploads `1_sample_data.py`, `2_train.py`, etc. to the Colab server
   - You do **NOT** need to upload the 124k images
5. Mount Google Drive (only needed to save results at the end)

**Data source:** Dataset is downloaded from Kaggle inside Colab (~10 min). No image upload required.

## 0. Setup

In [ ]:
# SKIP THIS CELL FOR NOW — mount Drive in Section 6 before saving results.
# Or run it now if you already want Drive available:
# from google.colab import drive
# drive.mount('/content/drive')
print('Drive mount skipped. Run the mount cell in Section 6 before saving results.')

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — change runtime!')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# Install dependencies
!pip install -q ultralytics scikit-learn matplotlib kagglehub
!pip install -q grad-cam   # for Grad-CAM visualization

import os, sys, shutil, glob
from pathlib import Path

WORK_DIR = Path('/content/iris-work')
WORK_DIR.mkdir(exist_ok=True)

# --- Find scripts on the Colab server ---
# Option 1: uploaded via VS Code "Upload to Colab" (usually lands in /content)
SCRIPT_SOURCES = [
    Path('/content/vinay-run'),
    Path('/content/IRIS/vinay-run'),
    Path.cwd(),
    Path('/content/drive/MyDrive/IRIS-data/vinay-run'),  # fallback: Drive
]

scripts_found = False
for src in SCRIPT_SOURCES:
    py_files = list(src.glob('*.py'))
    if py_files and (src / '1_sample_data.py').exists():
        for f in py_files:
            shutil.copy(f, WORK_DIR / f.name)
        print(f'Scripts copied from: {src}')
        scripts_found = True
        break

if not scripts_found:
    raise FileNotFoundError(
        'Could not find vinay-run scripts on Colab server.\n'
        'Fix: In VS Code, right-click vinay-run/ folder → Upload to Colab, then re-run this cell.'
    )

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
print('Working directory:', WORK_DIR)
print('Scripts:', sorted(p.name for p in WORK_DIR.glob('*.py')))

## 1. Download Dataset from Kaggle
Downloads the full dataset to the Colab server (~10 min). No upload needed.

In [ ]:
# Download kellysanderson/myopia-image-dataset from Kaggle
import kagglehub
from pathlib import Path

print('Downloading dataset from Kaggle (first run may take ~10 min)...')
dataset_path = kagglehub.dataset_download('kellysanderson/myopia-image-dataset')
print(f'Dataset downloaded to: {dataset_path}')

IMAGES_DIR = Path(dataset_path) / 'IMAGES'
print(f'IMAGES_DIR: {IMAGES_DIR}')
print(f'Myopia images:  {len(list((IMAGES_DIR / "Myopia_images").glob("*"))):,}')
print(f'Normal images:  {len(list((IMAGES_DIR / "Normal_images").glob("*"))):,}')

## 2. Build Data Splits
Randomly samples 5,000 per class from the downloaded dataset. Skip the 20k cell if short on time.

In [3]:
# Build 10k balanced split (5,000 Myopia + 5,000 Normal)
!python 1_sample_data.py --images-dir "{IMAGES_DIR}" --output-dir data_10k --n-per-class 5000


Dataset created at: data_10k
Mode: raw (no normalization)
Seed: 42  |  Images per class: 5000
Split ratios: train=0.7 / val=0.15 / test=0.15

Class           Train      Val     Test    Total
--------------------------------------------
Myopia           3500      750      750     5000
Normal           3500      750      750     5000
--------------------------------------------
TOTAL            7000     1500     1500    10000

Done. 10000 images copied.


In [4]:
# Verify split counts
for split in ['train', 'validation', 'test']:
    for cls in ['Myopia', 'Normal']:
        p = Path('data_10k') / split / cls
        count = len(list(p.glob('*')))
        print(f'data_10k/{split}/{cls}: {count}')

data_10k/train/Myopia: 3500
data_10k/train/Normal: 3500
data_10k/validation/Myopia: 750
data_10k/validation/Normal: 750
data_10k/test/Myopia: 750
data_10k/test/Normal: 750


In [ ]:
# OPTIONAL — Build 20k split for run_D (skip if short on time)
!python 1_sample_data.py --images-dir "{IMAGES_DIR}" --output-dir data_20k --n-per-class 10000

## 2. (Optional) CLAHE Brightness Normalization Experiment

The dataset has a **systematic brightness gap**: Myopia images are ~38 brightness
points brighter than Normal on average (102 vs 64 out of 255). A simple threshold
classifier achieves ~92% accuracy using only brightness — the model may not be
learning genuine retinal anatomy.

**CLAHE** equalizes brightness locally per image, removing this class-level bias.

- Accuracy **drops significantly** after CLAHE → model was using brightness shortcut  
- Accuracy **stays similar** → model learned real anatomical features too  

This becomes **run_E** in your results table. Skip to Section 3 if short on time.

In [ ]:
# Build CLAHE-normalized 10k split (~15-20 min — applies equalization to every image)
!pip install -q opencv-python-headless
!python 1_sample_data.py --images-dir "{IMAGES_DIR}" --output-dir data_10k_clahe --n-per-class 5000 --clahe

In [ ]:
# run_E — same config as run_B (YOLOv8n, 224px) but on CLAHE-normalized data
!python 2_train.py \
    --data data_10k_clahe \
    --model yolov8n-cls.pt \
    --imgsz 224 \
    --epochs 50 \
    --batch 32 \
    --name run_E

In [ ]:
# Evaluate run_E on CLAHE test set
!python 3_evaluate.py \
    --run-dir runs/classify/run_E \
    --test-dir data_10k_clahe/test \
    --output-dir .

# KEY QUESTION: Does accuracy drop vs run_B (99.9%)?
# If yes (e.g. drops to 70-85%) -> brightness shortcut was real
# If no (stays near 99%) -> model learned anatomical features too

## 3. Train (Main Experiments)

In [5]:
# Run A — Reproduce paper baseline (64px, YOLOv8n, 20 epochs)
!python 2_train.py \
    --data data_10k \
    --model yolov8n-cls.pt \
    --imgsz 64 \
    --epochs 20 \
    --batch 32 \
    --name run_A

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

  Run:    run_A
  Model:  yolov8n-cls.pt
  imgsz:  64
  epochs: 20  (patience=15)
  batch:  32
  data:   /content/iris-work/data_10k

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/iris-work/data_10k, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, ex

In [8]:
# Run B — Resolution upgrade (224px, YOLOv8n, 50 epochs)
!python 2_train.py \
    --data data_10k \
    --model yolov8n-cls.pt \
    --imgsz 224 \
    --epochs 50 \
    --batch 32 \
    --name run_B


  Run:    run_B
  Model:  yolov8n-cls.pt
  imgsz:  224
  epochs: 50  (patience=15)
  batch:  32
  data:   /content/iris-work/data_10k

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/iris-work/data_10k, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937,

In [11]:
# Run C — Model + resolution upgrade (224px, YOLOv8s, 50 epochs)  ← LIKELY BEST
!python 2_train.py \
    --data data_10k \
    --model yolov8s-cls.pt \
    --imgsz 224 \
    --epochs 50 \
    --batch 32 \
    --name run_C


  Run:    run_C
  Model:  yolov8s-cls.pt
  imgsz:  224
  epochs: 50  (patience=15)
  batch:  32
  data:   /content/iris-work/data_10k

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/iris-work/data_10k, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937,

In [ ]:
# Run D — More data (224px, YOLOv8s, 50 epochs, 20k images)
# Only run this if you have time left in the session
!python 2_train.py \
    --data data_20k \
    --model yolov8s-cls.pt \
    --imgsz 224 \
    --epochs 50 \
    --batch 32 \
    --name run_D

## 3. Evaluate All Runs on the Test Set
Always evaluate on the held-out test set, not validation.

In [6]:
# Evaluate Run A
!python 3_evaluate.py \
    --run-dir runs/classify/run_A \
    --test-dir data_10k/test \
    --output-dir .


Loading model: /content/iris-work/runs/classify/run_A/weights/best.pt
Evaluating on test set: /content/iris-work/data_10k/test
Classes found: ['Myopia', 'Normal']
  Evaluating Myopia: 750 images...
  Evaluating Normal: 750 images...

  Results for: run_A
  Accuracy:  0.9967  (99.7%)
  Precision: 0.9947  (99.5%)
  Recall:    0.9987  (99.9%)
  F1 Score:  0.9967  (99.7%)
  ROC AUC:   0.9998
  Avg Prec:  0.9998

              precision    recall  f1-score   support

      Normal       1.00      0.99      1.00       750
      Myopia       0.99      1.00      1.00       750

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500

Confusion Matrix (rows=True, cols=Predicted):
  Labels: [Normal, Myopia]
[[746   4]
 [  1 749]]

  TP (Myopia correctly identified): 749
  TN (Normal correctly identified): 746
  FP (Normal wrongly flagged as Myopia): 4
  FN (Myopia missed): 1

Figures saved t

In [13]:
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DEST = Path('/content/drive/MyDrive/IRIS-data/vinay-run-results')
DEST.mkdir(parents=True, exist_ok=True)

# Save the full run_X folder (weights + plots + results.csv)
shutil.copytree('runs/classify/run_C', DEST / 'run_C', dirs_exist_ok=True)
print('run_C saved:', DEST / 'run_C')

# Save evaluation figures
if Path('figures/run_C').exists():
    shutil.copytree('figures/run_C', DEST / 'figures' / 'run_C', dirs_exist_ok=True)
    print('Figures saved')

# Save results CSV
if Path('results/results_summary.csv').exists():
    shutil.copy('results/results_summary.csv', DEST / 'results_summary.csv')
    print('CSV saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
run_C saved: /content/drive/MyDrive/IRIS-data/vinay-run-results/run_C
Figures saved
CSV saved


In [9]:
# Evaluate Run B
!python 3_evaluate.py \
    --run-dir runs/classify/run_B \
    --test-dir data_10k/test \
    --output-dir .


Loading model: /content/iris-work/runs/classify/run_B/weights/best.pt
Evaluating on test set: /content/iris-work/data_10k/test
Classes found: ['Myopia', 'Normal']
  Evaluating Myopia: 750 images...
  Evaluating Normal: 750 images...

  Results for: run_B
  Accuracy:  0.9980  (99.8%)
  Precision: 0.9960  (99.6%)
  Recall:    1.0000  (100.0%)
  F1 Score:  0.9980  (99.8%)
  ROC AUC:   1.0000
  Avg Prec:  1.0000

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       750
      Myopia       1.00      1.00      1.00       750

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500

Confusion Matrix (rows=True, cols=Predicted):
  Labels: [Normal, Myopia]
[[747   3]
 [  0 750]]

  TP (Myopia correctly identified): 750
  TN (Normal correctly identified): 747
  FP (Normal wrongly flagged as Myopia): 3
  FN (Myopia missed): 0

Figures saved 

In [12]:
# Evaluate Run C
!python 3_evaluate.py \
    --run-dir runs/classify/run_C \
    --test-dir data_10k/test \
    --output-dir .


Loading model: /content/iris-work/runs/classify/run_C/weights/best.pt
Evaluating on test set: /content/iris-work/data_10k/test
Classes found: ['Myopia', 'Normal']
  Evaluating Myopia: 750 images...
  Evaluating Normal: 750 images...

  Results for: run_C
  Accuracy:  0.9973  (99.7%)
  Precision: 0.9947  (99.5%)
  Recall:    1.0000  (100.0%)
  F1 Score:  0.9973  (99.7%)
  ROC AUC:   0.9999
  Avg Prec:  0.9999

              precision    recall  f1-score   support

      Normal       1.00      0.99      1.00       750
      Myopia       0.99      1.00      1.00       750

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500

Confusion Matrix (rows=True, cols=Predicted):
  Labels: [Normal, Myopia]
[[746   4]
 [  0 750]]

  TP (Myopia correctly identified): 750
  TN (Normal correctly identified): 746
  FP (Normal wrongly flagged as Myopia): 4
  FN (Myopia missed): 0

Figures saved 

In [ ]:
# Evaluate Run D (if trained)
!python 3_evaluate.py \
    --run-dir runs/classify/run_D \
    --test-dir data_20k/test \
    --output-dir .

In [ ]:
# Print summary comparison table
import csv
from pathlib import Path

csv_path = Path('results/results_summary.csv')
if csv_path.exists():
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    
    print(f"{'Run':<10} {'Acc':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'AUC':>8}")
    print('-' * 55)
    for r in rows:
        print(f"{r['run']:<10} {float(r['accuracy'])*100:>7.1f}% {float(r['precision'])*100:>7.1f}% "
              f"{float(r['recall'])*100:>7.1f}% {float(r['f1'])*100:>7.1f}% {float(r['roc_auc']):>8.3f}")
else:
    print('No results yet — run evaluation cells above first.')

## 4. Threshold Optimization
Run on whichever experiment produced the best AUC.

In [ ]:
# Change run_C to whichever run had the best AUC
BEST_RUN = 'run_C'

!python 4_threshold_sweep.py \
    --run-dir runs/classify/{BEST_RUN} \
    --test-dir data_10k/test \
    --output-dir .

In [ ]:
# Display threshold sweep plot
from IPython.display import Image as IPImage
IPImage(f'figures/{BEST_RUN}/threshold_sweep.png')

## 5. Grad-CAM Visualization

In [ ]:
!python 5_gradcam.py \
    --run-dir runs/classify/{BEST_RUN} \
    --test-dir data_10k/test \
    --output-dir . \
    --method gradcam

In [ ]:
# Display Grad-CAM figure
from IPython.display import Image as IPImage
IPImage(f'figures/{BEST_RUN}/gradcam.png')

In [ ]:
# Mount Google Drive to save results
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/IRIS-data')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Results will save to:', DRIVE_ROOT / 'vinay-run-results')

## 6. Save Everything Back to Drive

In [ ]:
import shutil
from pathlib import Path

from runs_utils import resolve_run_dir

SAVE_DEST = DRIVE_ROOT / 'vinay-run-results'
SAVE_DEST.mkdir(exist_ok=True)

# Save weights for the best run
run_dir = resolve_run_dir(Path('runs') / BEST_RUN)
weights_src = run_dir / 'weights' / 'best.pt'
weights_dst = SAVE_DEST / f'{BEST_RUN}_best.pt'
shutil.copy(weights_src, weights_dst)
print(f'Weights saved: {weights_dst}')

# Save figures
figs_src = Path('figures')
figs_dst = SAVE_DEST / 'figures'
if figs_src.exists():
    shutil.copytree(figs_src, figs_dst, dirs_exist_ok=True)
    print(f'Figures saved: {figs_dst}')

# Save results CSV
csv_src = Path('results/results_summary.csv')
if csv_src.exists():
    shutil.copy(csv_src, SAVE_DEST / 'results_summary.csv')
    print(f'Results CSV saved: {SAVE_DEST / "results_summary.csv"}')

# Save training results.csv for each run
for search_root in [Path('runs'), Path('runs/classify'), Path('runs/classify/runs/classify')]:
    if not search_root.exists():
        continue
    for run_dir in search_root.iterdir():
        if not run_dir.is_dir():
            continue
        r = run_dir / 'results.csv'
        if r.exists():
            shutil.copy(r, SAVE_DEST / f'{run_dir.name}_training_curves.csv')

print('\nAll outputs saved to Google Drive.')

---
## Results Table for Paper
Fill in after running all evaluations.

In [ ]:
# Auto-generate LaTeX table from results CSV
import csv
from pathlib import Path

LABELS = {
    'run_A': 'YOLOv8n / 64px / 10k (baseline)',
    'run_B': 'YOLOv8n / 224px / 10k',
    'run_C': 'YOLOv8s / 224px / 10k',
    'run_D': 'YOLOv8s / 224px / 20k',
    'run_E': 'YOLOv8n / 224px / 10k + CLAHE (brightness normalized)',
}

csv_path = Path('results/results_summary.csv')
if csv_path.exists():
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    
    print('\\begin{table}[h]')
    print('\\centering')
    print('\\begin{tabular}{lcccccc}')
    print('\\hline')
    print('Configuration & Accuracy & Precision & Recall & F1 & AUC \\\\')
    print('\\hline')
    for r in rows:
        label = LABELS.get(r['run'], r['run'])
        print(f"{label} & {float(r['accuracy'])*100:.1f}\\% & {float(r['precision'])*100:.1f}\\% & "
              f"{float(r['recall'])*100:.1f}\\% & {float(r['f1'])*100:.1f}\\% & {float(r['roc_auc']):.3f} \\\\")
    print('\\hline')
    print('\\end{tabular}')
    print('\\caption{Model comparison across configurations}')
    print('\\end{table}')
else:
    print('Run evaluation cells first.')